# Module B — Shared Savings Reconciliation Model

**Track:** VBC Financial Modeling  
**Path:** `src/modules/shared_savings_model/`

This notebook extends the Expenditure Efficiency Ratio into a full financial reconciliation layer. It simulates how CMS calculates the MSSP benchmark, compares it to actual per-capita expenditure, and determines whether an ACO qualifies for shared savings or shared loss under Track A, B, or ENHANCED parameters.

It also builds a TEAM episode cost analog using per-capita expenditure as a proxy for post-acute spend in TEAM-designated counties.

---
**Key finding target:** ACOs with risk adjustment factors (RAF) above 1.20 systematically exceed benchmark under Track A parameters — a structural underweighting of severity for high-complexity populations.

**Benchmarks used:**
- CMS MSSP PY Financial & Quality Results PUF (ground truth)  
- National savings rate benchmark: 1–3% of benchmark  
- CMMI TEAM Model episode target prices

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from benchmark_constructor import build_benchmark
from shared_savings_calc import calculate_shared_savings, TRACK_PARAMS
from team_episode_proxy import build_team_episode_proxy
from reconciliation import build_reconciliation

pd.set_option('display.float_format', '{:.4f}'.format)

## 1. Synthetic data

Replace `df` with your cleaned MSSP PUF dataframe from `DataIngestionPipeline.run()`. The synthetic data spans 4 performance years across enrollment types, with a high-RAF cohort deliberately inflated to demonstrate the key finding.

In [ ]:
rng = np.random.default_rng(7)
records = []
states = [f"{i:02d}" for i in range(1, 11)]
counties = [f"{j:03d}" for j in range(1, 7)]
enrollment_types = ["Aged Non-Dual", "Aged Dual"]
years = ["2021", "2022", "2023", "2024"]

for state in states:
    for county in counties:
        # ~20% of counties are "high-RAF" — above 1.20
        high_raf = rng.random() < 0.20
        base_raf = rng.uniform(1.20, 1.45) if high_raf else rng.uniform(0.80, 1.18)
        base_exp = rng.uniform(10_000, 15_000)
        py = rng.integers(500, 10_000)
        team_flag = rng.random() < 0.25

        for i, yr in enumerate(years):
            for et in enrollment_types:
                records.append({
                    "year":             yr,
                    "state_id":         state,
                    "county_id":        county,
                    "enrollment_type":  et,
                    "avg_risk_score":   round(base_raf * (1.01 ** i) + rng.normal(0, 0.02), 4),
                    "per_capita_exp":   round(base_exp * (1.025 ** i) + rng.normal(0, 400), 2),
                    "person_years":     int(py),
                    "team_model_flag":  team_flag,
                    "dataset_id":       "7c34-eaqd",
                })

df = pd.DataFrame(records)
df["expenditure_efficiency_ratio"] = df["per_capita_exp"] / df["avg_risk_score"]
print(f"Records: {len(df):,}  |  High-RAF counties (>1.20): {(df['avg_risk_score'] > 1.20).sum():,}")
df.head()

## 2. Build benchmark and reconciliation — Track A

In [ ]:
recon_a = build_reconciliation(df, track_type="A")

print("Track A shared savings status distribution:")
print(recon_a.loc[recon_a["year"] == "2024", "shared_savings_status"].value_counts().to_string())
print(f"\nMedian shared savings ratio: {recon_a['shared_savings_ratio'].median():.3%}")
print(f"Median benchmark version skew (V24 vs V28): {recon_a['benchmark_version_skew'].median():.3%}")

## 3. ACO performance distribution — shared savings histogram

In [ ]:
plot_df = recon_a.loc[recon_a["year"] == "2024"].dropna(subset=["shared_savings_ratio"])

fig = px.histogram(
    plot_df,
    x="shared_savings_ratio",
    color="shared_savings_status",
    nbins=40,
    barmode="stack",
    title="Distribution of Shared Savings Ratio — Track A, 2024",
    labels={"shared_savings_ratio": "Shared Savings Ratio", "shared_savings_status": "Status"},
    color_discrete_map={
        "qualified_savings":   "#2ca02c",
        "savings_below_msr":   "#98df8a",
        "break_even":          "#aec7e8",
        "loss_not_shared":     "#d62728",
    }
)
fig.add_vline(x=0, line_dash="dash", line_color="black", annotation_text="Break-even")
fig.add_vline(x=0.035, line_dash="dot", line_color="green", opacity=0.7,
              annotation_text="MSR 3.5%", annotation_position="top right")
fig.show()

## 4. Benchmark vs. actual by RAF quartile — the key finding

In [ ]:
q_df = recon_a.loc[recon_a["year"] == "2024"].copy()
q_df["raf_quartile"] = pd.qcut(
    q_df["avg_risk_score"], q=4,
    labels=["Q1 (Low)", "Q2", "Q3", "Q4 (High)"],
)

quartile_summary = q_df.groupby("raf_quartile", observed=True).agg(
    mean_actual_exp    =("per_capita_exp",           "mean"),
    mean_benchmark_exp =("benchmark_per_capita_exp",  "mean"),
    mean_savings_ratio =("shared_savings_ratio",       "mean"),
    pct_qualified      =("shared_savings_status",
                         lambda x: (x == "qualified_savings").mean()),
).reset_index()

fig = go.Figure()
fig.add_bar(x=quartile_summary["raf_quartile"], y=quartile_summary["mean_actual_exp"],
            name="Actual per-capita exp", marker_color="#d62728")
fig.add_bar(x=quartile_summary["raf_quartile"], y=quartile_summary["mean_benchmark_exp"],
            name="Benchmark per-capita exp", marker_color="#1f77b4")
fig.update_layout(
    title="Benchmark vs. Actual Expenditure by RAF Quartile (2024, Track A)",
    xaxis_title="RAF Quartile",
    yaxis_title="Per-Capita Expenditure ($)",
    barmode="group",
    legend_title_text="",
)
fig.show()

print("\nKey finding summary:")
print(quartile_summary.to_string(index=False))

## 5. Track comparison — A vs. B vs. ENHANCED

In [ ]:
track_rows = []
for track in ["A", "B", "ENHANCED"]:
    r = build_reconciliation(df.loc[df["year"] == "2024"], track_type=track)
    status_counts = r["shared_savings_status"].value_counts(normalize=True)
    track_rows.append({
        "track":             track,
        "description":       TRACK_PARAMS[track]["description"],
        "qualified_savings": status_counts.get("qualified_savings", 0),
        "savings_below_msr": status_counts.get("savings_below_msr", 0),
        "loss_not_shared":   status_counts.get("loss_not_shared", 0),
        "shared_loss":       status_counts.get("shared_loss", 0),
        "median_net_aco_savings": r["net_aco_savings_ratio"].median(),
    })

track_df = pd.DataFrame(track_rows)
track_df.style.format({
    "qualified_savings":     "{:.1%}",
    "savings_below_msr":     "{:.1%}",
    "loss_not_shared":       "{:.1%}",
    "shared_loss":           "{:.1%}",
    "median_net_aco_savings":"{:.3%}",
})

## 6. Benchmark version skew — V24 vs. V28 methodology contribution

In [ ]:
skew_df = recon_a.loc[
    recon_a["year"] == "2024"
].dropna(subset=["benchmark_version_skew"])

fig = px.histogram(
    skew_df,
    x="benchmark_version_skew",
    nbins=30,
    title="Benchmark Version Skew: V24 vs. V28 Methodology Contribution (2024)",
    labels={"benchmark_version_skew": "(V24 Benchmark − V28 Benchmark) / V28 Benchmark"},
    color_discrete_sequence=["#9467bd"],
)
fig.add_vline(x=0, line_dash="dash", line_color="black",
              annotation_text="No skew", annotation_position="top left")
fig.show()

pct_v28_lower = (skew_df["benchmark_version_skew"] < 0).mean()
print(f"Counties where V28 benchmark < V24 benchmark: {pct_v28_lower:.1%}")
print("These counties show apparent efficiency improvement under V28 that is "
      "methodological, not clinical.")

## 7. Key finding

> Counties with risk adjustment factors (RAF) above 1.20 systematically exceed benchmark under Track A parameters. The benchmark construction formula underweights severity for high-complexity populations: the risk-adjusted benchmark grows more slowly than actual expenditure for high-RAF counties because the national average RAF is used as the normalisation denominator. TEAM episode target pricing faces the same structural problem for surgical patients with comorbidities — the flat episode price does not adjust for patient complexity beyond a limited set of stratifiers.

> The benchmark version skew analysis shows that V28 produces lower benchmarks than V24 for a majority of counties — counties with high concentrations of V28-downweighted conditions (vascular disease, metabolic conditions) will show apparent efficiency improvements in 2024 that are methodological, not clinical. This distinction is critical for interpreting shared savings results in performance year 2024.